# 🌾 Bori V3 — Control Panel

**Bilingual (Korean-English) SLM** adapted from SmolLM2 via CPT + SFT.

This notebook is a **thin control panel**. All logic lives in the codebase (uploaded as a Kaggle dataset input).

| Feature | Details |
|---|---|
| Base Model | SmolLM2-135M or SmolLM2-360M |
| Vocab | 58,133 (merged Korean BPE) |
| Scheduler | WSD (Warmup-Stable-Decay) |
| Attention | SDPA |
| SFT Loss | Response-only (prompt masked) |

## ⚙️ Configuration
Edit these variables to control the entire pipeline.

In [ ]:
# ══════════════════════════════════════════════════════════════
#                    BORI-3 CONFIGURATION
# ══════════════════════════════════════════════════════════════

# ── Model Selection ──
# Options: 'HuggingFaceTB/SmolLM2-135M', 'HuggingFaceTB/SmolLM2-360M'
MODEL_NAME = 'HuggingFaceTB/SmolLM2-360M'

# ── Run Identity ──
RUN_NAME = 'bori-v3-360m'
SEED = 42

# ── Phase Control ──
# Toggle which phases to run this session
RUN_TOKENIZER = False     # Train/merge Korean tokenizer (only needed once)
RUN_CPT       = True      # Continuous Pre-Training
RUN_SFT       = True      # Supervised Fine-Tuning
RUN_BENCHMARK = True      # Run evaluation benchmark
SKIP_PHASE1A  = False     # Skip embedding warmup (set True after first session)

# ── CPT Hyperparameters ──
CPT_LR          = 3e-4
CPT_SCHEDULER   = 'wsd'
CPT_MAX_STEPS   = 15000
CPT_WARMUP      = 300
CPT_BATCH_SIZE  = 2
CPT_GRAD_ACCUM  = 16
CPT_REPLAY_RATIO = 0.15
CPT_SAVE_STEPS  = 500

# ── SFT Hyperparameters ──
SFT_LR         = 5e-5
SFT_SCHEDULER  = 'cosine'
SFT_MAX_STEPS  = 3000
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 32
SFT_SAVE_STEPS = 200

# ── SFT Datasets ──
# 50% Korean, 50% English — balanced bilingual instruction following
SFT_DATASETS = ','.join([
    'brandonbaek/konglish-synthetic-instruct',  # Korean instructions
    'jojo0217/korean_safe_conversation',         # Korean conversational
    'maywell/ko_wikidata_QA',                    # Korean factual Q&A
    'HuggingFaceH4/no_robots',                   # English human-written instructions
    'yahma/alpaca-cleaned',                      # English diverse instructions
])
SFT_PROBS = '0.20,0.15,0.15,0.25,0.25'

# ── W&B ──
try:
    import wandb
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret('wandb_api_key'))
    WANDB_PROJECT = 'bori-v3'
    print('✅ W&B logged in.')
except Exception:
    WANDB_PROJECT = None
    print('W&B skipped. Training continues without logging.')

# ── Derived (don't edit) ──
WANDB_FLAG = f'--wandb_project {WANDB_PROJECT}' if WANDB_PROJECT else ''
WANDB_CPT_ID = f'--wandb_run_id {RUN_NAME}-cpt' if WANDB_PROJECT else ''
WANDB_SFT_ID = f'--wandb_run_id {RUN_NAME}-sft' if WANDB_PROJECT else ''

print(f'\n📋 Config Summary:')
print(f'   Model:     {MODEL_NAME}')
print(f'   Run:       {RUN_NAME}')
print(f'   CPT:       {CPT_MAX_STEPS} steps, LR={CPT_LR}, {CPT_SCHEDULER}')
print(f'   SFT:       {SFT_MAX_STEPS} steps, LR={SFT_LR}')
print(f'   Phases:    tokenizer={RUN_TOKENIZER} cpt={RUN_CPT} sft={RUN_SFT} bench={RUN_BENCHMARK}')


## 🔧 Step 0: Environment Setup
Installs deps, extracts codebase, restores checkpoints.

In [ ]:
import sys
!{sys.executable} -m pip install -q 'transformers>=4.40.0' 'accelerate>=0.29.0' 'datasets>=2.18.0' 'wandb>=0.16.0' 'tokenizers>=0.15.0' rich tqdm

# Run the setup script (extracts codebase, restores checkpoints, configures accelerate)
!{sys.executable} scripts/setup_kaggle.py --model_name {MODEL_NAME}

# Detect GPU config
import torch, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
num_gpus = torch.cuda.device_count()
accelerate_config = 'configs/accelerate_config_multi.yaml' if num_gpus > 1 else 'configs/accelerate_config_single.yaml'
print(f'\n🖥️ {num_gpus} GPU(s) | Config: {accelerate_config}')


## 📝 Step 1: Tokenizer (skip if already merged)

In [ ]:
import os, sys
python_path = sys.executable

if RUN_TOKENIZER:
    if os.path.exists('data/merged_tokenizer/tokenizer.json'):
        print('✅ Merged tokenizer already exists. Skipping.')
    else:
        !{python_path} scripts/train_korean_tokenizer.py --vocab_size 10000 --num_samples 500000 --output_dir data/korean_tokenizer
        !{python_path} scripts/merge_tokenizers.py --base_model {MODEL_NAME} --korean_tokenizer_path data/korean_tokenizer --output_dir data/merged_tokenizer
else:
    print('⏭️ Tokenizer step skipped.')


## 🏋️ Step 2: Continuous Pre-Training (CPT)
Phase 1a: Embedding warmup (1k steps) → Phase 1b: Full CPT (15k steps, ~2B tokens)

In [ ]:
import sys
python_path = sys.executable
skip_flag = '--skip_phase1a' if SKIP_PHASE1A else ''

if RUN_CPT:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {python_path} -m accelerate.commands.launch \
        --config_file {accelerate_config} scripts/cpt.py \
        --model_name {MODEL_NAME} \
        --tokenizer_path data/merged_tokenizer \
        --dataset_name "HuggingFaceFW/fineweb-2:kor_Hang" \
        --replay_dataset "HuggingFaceFW/fineweb-edu-dedup" \
        --replay_ratio {CPT_REPLAY_RATIO} \
        --warmup_lr 1e-3 \
        --warmup_steps_phase1a 1000 \
        --cpt_lr {CPT_LR} \
        --lr_scheduler_type {CPT_SCHEDULER} \
        --warmup_steps {CPT_WARMUP} \
        --max_steps {CPT_MAX_STEPS} \
        --save_steps {CPT_SAVE_STEPS} \
        --batch_size {CPT_BATCH_SIZE} \
        --grad_accum {CPT_GRAD_ACCUM} \
        --max_seq_length 2048 \
        --output_dir /kaggle/working/cpt_checkpoints \
        {WANDB_FLAG} {WANDB_CPT_ID} \
        --run_name {RUN_NAME} {skip_flag} \
        --seed {SEED}
else:
    print('⏭️ CPT skipped.')


## 💬 Step 3: Supervised Fine-Tuning (SFT)
Response-only loss on curated bilingual instruction mix (~60% Korean, ~40% English).

In [ ]:
import sys, os
python_path = sys.executable

# Auto-resolve best available CPT model path
model_path = '/kaggle/working/cpt_checkpoints/phase_1b'
if not os.path.exists(os.path.join(model_path, 'config.json')):
    if os.path.exists(model_path):
        subdirs = sorted([d for d in os.listdir(model_path) if d.startswith('checkpoint-')],
                         key=lambda x: int(x.split('-')[1]), reverse=True)
        if subdirs:
            model_path = os.path.join(model_path, subdirs[0])
            print(f'Resolved CPT model: {model_path}')

if RUN_SFT:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True {python_path} -m accelerate.commands.launch \
        --config_file {accelerate_config} scripts/sft.py \
        --model_path {model_path} \
        --tokenizer_path data/merged_tokenizer \
        --dataset_name "{SFT_DATASETS}" \
        --dataset_probs "{SFT_PROBS}" \
        --max_steps {SFT_MAX_STEPS} \
        --learning_rate {SFT_LR} \
        --lr_scheduler_type {SFT_SCHEDULER} \
        --batch_size {SFT_BATCH_SIZE} \
        --grad_accum {SFT_GRAD_ACCUM} \
        --save_steps {SFT_SAVE_STEPS} \
        --output_dir /kaggle/working/sft_checkpoints \
        {WANDB_FLAG} {WANDB_SFT_ID} \
        --run_name {RUN_NAME}-sft \
        --seed {SEED}
else:
    print('⏭️ SFT skipped.')


## 📊 Step 4: Benchmark & Evaluation

In [ ]:
import sys, os
python_path = sys.executable

# Auto-resolve best available model for benchmarking
bench_model = '/kaggle/working/sft_checkpoints'
if not os.path.exists(os.path.join(bench_model, 'config.json')):
    if os.path.exists(bench_model):
        subdirs = sorted([d for d in os.listdir(bench_model) if d.startswith('checkpoint-')],
                         key=lambda x: int(x.split('-')[1]), reverse=True)
        if subdirs:
            bench_model = os.path.join(bench_model, subdirs[0])
    else:
        bench_model = '/kaggle/working/cpt_checkpoints/phase_1b'
print(f'Benchmarking model: {bench_model}')

if RUN_BENCHMARK:
    !{python_path} scripts/benchmark.py \
        --model_path {bench_model} \
        --tokenizer_path data/merged_tokenizer \
        --output_file /kaggle/working/benchmark_results.json \
        --num_perplexity_samples 50
else:
    print('⏭️ Benchmark skipped.')


## 📦 Step 5: Export Checkpoints

In [ ]:
import shutil, os

print('Zipping checkpoints...')
if os.path.exists('/kaggle/working/cpt_checkpoints/phase_1b'):
    shutil.make_archive('/kaggle/working/bori_v3_cpt_export', 'zip', '/kaggle/working/cpt_checkpoints/phase_1b')
    print('✅ CPT exported.')

if os.path.exists('/kaggle/working/sft_checkpoints'):
    shutil.make_archive('/kaggle/working/bori_v3_sft_export', 'zip', '/kaggle/working/sft_checkpoints')
    print('✅ SFT exported.')

print('\nAll done! Download zips from the Output tab.')
